# TASK 1: Language Translation Tool



In [5]:
!pip install click==8.2.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.2/102.2 kB 2.6 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.1.8
    Uninstalling click-8.1.8:
      Successfully uninstalled click-8.1.8
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gtts 2.5.4 requires click<8.2,>=7.1, but you have click 8.2.1 which is incompatible.
rasterio 1.5.0 requires click!=8.2.*,>=4.0, but you have click 8.2.1 which is incompatible.


In [1]:
# Section 1: Install required libraries
!pip install deep-translator gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 1.5 MB/s eta 0:00:00


#  Section 2 — Import & Language Setup

In [1]:
# Section 2: Imports and language map
from deep_translator import GoogleTranslator
import gradio as gr

# Supported languages (label → code)
LANGUAGES = {
    "Auto Detect":   "auto",
    "English":       "en",
    "Urdu":          "ur",
    "Roman Urdu":    "ur",   # transliteration goes en→ur then display
    "French":        "fr",
    "German":        "de",
    "Spanish":       "es",
    "Arabic":        "ar",
    "Chinese (Simplified)": "zh-CN",
    "Japanese":      "ja",
    "Hindi":         "hi",
    "Turkish":       "tr",
    "Russian":       "ru",
    "Portuguese":    "pt",
    "Italian":       "it",
    "Korean":        "ko",
}

LANG_NAMES = list(LANGUAGES.keys())


* We build a human-readable dropdown list that maps
to BCP-47 language codes (what Google Translate actually uses).
* "auto" lets Google detect the source language automatically.


# Section 3 — Core Translation Function

In [2]:
# Section 3: Translation logic
def translate_text(text, source_lang_name, target_lang_name):
    """
    Takes raw text + language names, returns translated string or error.
    """
    if not text.strip():
        return "⚠️ Please enter some text to translate."

    source_code = LANGUAGES.get(source_lang_name, "auto")
    target_code = LANGUAGES.get(target_lang_name, "en")

    if source_code == target_code and source_code != "auto":
        return "ℹ️ Source and target languages are the same."

    try:
        translator = GoogleTranslator(source=source_code, target=target_code)
        result = translator.translate(text)
        return result if result else "⚠️ Translation returned empty. Try again."
    except Exception as e:
        return f"❌ Error: {str(e)}"

* Guards against empty input and same-language pairs.
* GoogleTranslator(source=..., target=...) creates a translator instance.
* .translate(text) fires the request to Google Translate (free tier, no key).
* Wrapped in try/except so API errors show cleanly instead of crashing.

#  Section 4 — Text-to-Speech (Optional Feature)

In [4]:
from gtts import gTTS
import tempfile, os

def speak_translation(translated_text, target_lang_name):

    if not translated_text or translated_text.startswith(("⚠️", "❌", "ℹ️")):
        return None

    lang_code = LANGUAGES.get(target_lang_name, "en")

    if lang_code == "auto":
        lang_code = "en"

    try:
        tts = gTTS(text=translated_text, lang=lang_code)

        tmp = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3")

        tts.save(tmp.name)

        return tmp.name

    except Exception:
        return None

# Section 5 — Build the Gradio UI

In [5]:
# Section 5: Gradio UI
def full_pipeline(text, source_lang, target_lang):
    translation = translate_text(text, source_lang, target_lang)
    audio_path  = speak_translation(translation, target_lang)
    return translation, audio_path

with gr.Blocks(
    title="🌍 Language Translator",
    theme=gr.themes.Soft(),
    css="""
        #title  { text-align: center; font-size: 2rem; margin-bottom: 0.2rem; }
        #sub    { text-align: center; color: #666; margin-bottom: 1.5rem; }
        .box    { border-radius: 12px !important; }
    """
) as demo:

    gr.Markdown("# 🌍 Language Translator", elem_id="title")
    gr.Markdown("Powered by Google Translate · Free · No API Key", elem_id="sub")

    with gr.Row():
        src_lang = gr.Dropdown(
            choices=LANG_NAMES, value="Auto Detect",
            label="🔤 Source Language"
        )
        tgt_lang = gr.Dropdown(
            choices=[l for l in LANG_NAMES if l != "Auto Detect"],
            value="Urdu",
            label="🎯 Target Language"
        )

    input_text = gr.Textbox(
        lines=5, placeholder="Type or paste text here...",
        label="📝 Input Text", elem_classes="box"
    )

    translate_btn = gr.Button("🔄 Translate", variant="primary", size="lg")

    output_text = gr.Textbox(
        lines=5, label="✅ Translated Text",
        interactive=True,          # user can copy-edit
        show_copy_button=True,     # built-in copy button ✅
        elem_classes="box"
    )

    audio_out = gr.Audio(label="🔊 Listen to Translation", type="filepath")

    translate_btn.click(
        fn=full_pipeline,
        inputs=[input_text, src_lang, tgt_lang],
        outputs=[output_text, audio_out]
    )

    gr.Examples(
        examples=[
            ["Hello, how are you?",       "Auto Detect", "Urdu"],
            ["میں ٹھیک ہوں، شکریہ",        "Auto Detect", "English"],
            ["The future belongs to AI.", "English",     "French"],
            ["人工知能は未来です",            "Japanese",    "English"],
        ],
        inputs=[input_text, src_lang, tgt_lang],
        label="💡 Try these examples"
    )

demo.launch(share=True)   # share=True gives a public URL in Colab

/tmp/ipykernel_3781/428016085.py:7: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_3781/428016085.py:7: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://26ec990b2d876ce0f8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
